<a href="https://colab.research.google.com/github/RachmadAprisandhy/PBL_Kelompok3/blob/Zacky/Klasifikasi_Grading_Pisang_CNN_Real.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Langkah 1 — Hubungkan Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

print("Google Drive berhasil dihubungkan.")

Mounted at /content/drive
Google Drive berhasil dihubungkan.


Langkah 2 — Tentukan folder Real Dataset

In [ ]:
import os

folder_dataset = (
    "/content/drive/MyDrive/Dataset PBL/"
    "Banana Images/Real Dataset"
)

print("Folder ditemukan:", os.path.isdir(folder_dataset))

print("\nIsi folder:")
if os.path.isdir(folder_dataset):
    print(sorted(os.listdir(folder_dataset)))

Folder ditemukan: True

Isi folder:
['test', 'train']


Langkah 3 — Periksa jumlah gambar setiap kelas

In [ ]:
import os

folder_dataset = (
    "/content/drive/MyDrive/Dataset PBL/"
    "Banana Images/Real Dataset"
)

kelas = ["Grade A", "Grade B", "Grade C", "Grade D"]
ekstensi = (".png", ".jpg", ".jpeg", ".bmp", ".webp")

total_semua = 0

for bagian in ["train", "test"]:
    folder_bagian = os.path.join(folder_dataset, bagian)
    total_bagian = 0

    print(f"\n===== {bagian.upper()} =====")

    for nama_kelas in kelas:
        folder_kelas = os.path.join(folder_bagian, nama_kelas)

        if not os.path.isdir(folder_kelas):
            print(f"{nama_kelas}: folder tidak ditemukan")
            continue

        jumlah = sum(
            1
            for nama in os.listdir(folder_kelas)
            if nama.lower().endswith(ekstensi)
        )

        total_bagian += jumlah
        print(f"{nama_kelas}: {jumlah} gambar")

    print(f"Total {bagian}: {total_bagian} gambar")
    total_semua += total_bagian

print("\n==============================")
print("TOTAL SELURUH DATA:", total_semua)
print("==============================")



===== TRAIN =====
Grade A: 375 gambar
Grade B: 375 gambar
Grade C: 375 gambar
Grade D: 375 gambar
Total train: 1500 gambar

===== TEST =====
Grade A: 125 gambar
Grade B: 125 gambar
Grade C: 125 gambar
Grade D: 125 gambar
Total test: 500 gambar

TOTAL SELURUH DATA: 2000


Langkah 4 — Preprocessing dan Memuat Dataset (224 × 224)

In [ ]:
import tensorflow as tf
import os

folder_dataset = (
    "/content/drive/MyDrive/Dataset PBL/"
    "Banana Images/Real Dataset"
)

ukuran_gambar = (224, 224)
batch_size = 16
seed = 42

# Memuat data training
train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(folder_dataset, "train"),
    labels="inferred",
    label_mode="int",
    image_size=ukuran_gambar,
    batch_size=batch_size,
    shuffle=True,
    seed=seed
)

# Memuat data testing
test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(folder_dataset, "test"),
    labels="inferred",
    label_mode="int",
    image_size=ukuran_gambar,
    batch_size=batch_size,
    shuffle=False
)

# Menyimpan urutan kelas
nama_kelas = train_ds.class_names

print("\nUrutan kelas:", nama_kelas)
print("Jumlah kelas:", len(nama_kelas))

# Normalisasi piksel menjadi rentang 0–1
train_ds = train_ds.map(
    lambda gambar, label: (
        tf.cast(gambar, tf.float32) / 255.0,
        label
    )
)

test_ds = test_ds.map(
    lambda gambar, label: (
        tf.cast(gambar, tf.float32) / 255.0,
        label
    )
)

# Mengoptimalkan proses input data
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

print("\nPreprocessing selesai.")


Found 1500 files belonging to 4 classes.
Found 500 files belonging to 4 classes.

Urutan kelas: ['Grade A', 'Grade B', 'Grade C', 'Grade D']
Jumlah kelas: 4

Preprocessing selesai.


Langkah 5 — Memeriksa Bentuk dan Ukuran Dataset

In [ ]:
# Mengambil satu batch dari dataset training dan testing
for gambar, label in train_ds.take(1):
    print("===== DATA TRAINING =====")
    print("Bentuk gambar:", gambar.shape)
    print("Bentuk label :", label.shape)
    print("Tipe data    :", gambar.dtype)
    print("Nilai piksel :", float(tf.reduce_min(gambar)),
          "sampai", float(tf.reduce_max(gambar)))

for gambar, label in test_ds.take(1):
    print("\n===== DATA TESTING =====")
    print("Bentuk gambar:", gambar.shape)
    print("Bentuk label :", label.shape)

print("\nUrutan kelas:", nama_kelas)
print("Jumlah kelas:", len(nama_kelas))

===== DATA TRAINING =====
Bentuk gambar: (16, 224, 224, 3)
Bentuk label : (16,)
Tipe data    : <dtype: 'float32'>
Nilai piksel : 0.0 sampai 1.0

===== DATA TESTING =====
Bentuk gambar: (16, 224, 224, 3)
Bentuk label : (16,)

Urutan kelas: ['Grade A', 'Grade B', 'Grade C', 'Grade D']
Jumlah kelas: 4


Langkah 6 — Membangun Arsitektur CNN

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Membuat arsitektur CNN baru
model = models.Sequential([
    # Input gambar 224 x 224 x 3 (RGB)
    layers.Input(shape=(224, 224, 3)),

    # Blok CNN 1
    layers.Conv2D(
        32, (3, 3),
        activation="relu",
        padding="same"
    ),
    layers.MaxPooling2D((2, 2)),

    # Blok CNN 2
    layers.Conv2D(
        64, (3, 3),
        activation="relu",
        padding="same"
    ),
    layers.MaxPooling2D((2, 2)),

    # Blok CNN 3
    layers.Conv2D(
        128, (3, 3),
        activation="relu",
        padding="same"
    ),
    layers.MaxPooling2D((2, 2)),

    # Merangkum fitur gambar
    layers.GlobalAveragePooling2D(),

    # Klasifikasi
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(4, activation="softmax")
])

# Menampilkan arsitektur model
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 101,764 (397.52 KB)

 Trainable params: 101,764 (397.52 KB)

 Non-trainable params: 0 (0.00 B)

Langkah 7 — Compile Model CNN

In [ ]:
# Compile model CNN
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model berhasil di-compile.")
print("Optimizer : Adam")
print("Loss      : Sparse Categorical Crossentropy")
print("Metric    : Accuracy")

Model berhasil di-compile.
Optimizer : Adam
Loss      : Sparse Categorical Crossentropy
Metric    : Accuracy


Langkah 8 — Training Model CNN

In [ ]:
# Menentukan jumlah epoch
jumlah_epoch = 20

# Melatih model CNN
history = model.fit(
    train_ds,
    epochs=jumlah_epoch,
    verbose=1
)

print("\nTraining model CNN selesai.")

Epoch 1/20
94/94 ━━━━━━━━━━━━━━━━━━━━ 200s 2s/step - accuracy: 0.3647 - loss: 1.2936
Epoch 2/20
94/94 ━━━━━━━━━━━━━━━━━━━━ 171s 2s/step - accuracy: 0.5667 - loss: 0.9784
Epoch 3/20
94/94 ━━━━━━━━━━━━━━━━━━━━ 167s 2s/step - accuracy: 0.6993 - loss: 0.7412
Epoch 4/20
94/94 ━━━━━━━━━━━━━━━━━━━━ 165s 2s/step - accuracy: 0.7367 - loss: 0.6385
Epoch 5/20
94/94 ━━━━━━━━━━━━━━━━━━━━ 202s 2s/step - accuracy: 0.7673 - loss: 0.5519
Epoch 6/20
94/94 ━━━━━━━━━━━━━━━━━━━━ 167s 2s/step - accuracy: 0.7713 - loss: 0.5546
Epoch 7/20
94/94 ━━━━━━━━━━━━━━━━━━━━ 166s 2s/step - accuracy: 0.7740 - loss: 0.5373
Epoch 8/20
94/94 ━━━━━━━━━━━━━━━━━━━━ 163s 2s/step - accuracy: 0.7767 - loss: 0.5233
Epoch 9/20
94/94 ━━━━━━━━━━━━━━━━━━━━ 174s 2s/step - accuracy: 0.7940 - loss: 0.4824
Epoch 10/20
94/94 ━━━━━━━━━━━━━━━━━━━━ 165s 2s/step - accuracy: 0.7947 - loss: 0.4724
Epoch 11/20
12/94 ━━━━━━━━━━━━━━━━━━━━ 2:25 2s/step - accuracy: 0.8176 - loss: 0.4838